In [2]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage,HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

<!--in core principle of this state Graph is during execution any node can access and modify the state but when execution ends the value in state or state got erased . 

So,To persist that store we need persistence concept.
-->

Persistence doesnot store the final value of state but also the intermediate values of the state.
It can save and also restore the state also.

Checkpointer in Persistence
-----------------------------------
in each step we can define checkpoint so fro each checkpoint persistence will store the state against that checkpoint.

Threads in Persistence
-------------------------------------
we can define one threadId so the persistence will store the state(with intermediates) against that thread.

for a user we can define conversation to one thread id.so when we will resume the chat afetr some days we will fetch previous state with that user and threadId(conversation)

In [3]:
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model="models/gemini-3.7-flash",
    temperature=0.7
)

In [15]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str

In [ ]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content[0]['text']

    return {'joke': response}

In [ ]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content[0]['text'] 

    return {'explanation': response}

In [18]:
graph = StateGraph(JokeState)
graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)
graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [4]:
result = llm.invoke("generate a joke on the topic on Pizza")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [11]:
result.content[0]['text']

'Why did the pizza get hired for the job?\n\nBecause it always **delivers**, works well with **dough**, and knows how to handle the **heat**!'

In [5]:
config = {"configurable":{"thread_id":1}}
workflow.invoke({'topic':"Pizza"},config=config)

NameError: name 'workflow' is not defined

Benifit of Persistence
---------------------------
1.Short Term Memory
2.Fault Tolerance
3.Human in the loop
4.Time travel

In [ ]:
# Time Travel
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})


workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})
# above will resume the workflow from previous step(checkpointId against that step's state )


list(workflow.get_state_history(config1))
# getting all the states


# Updating State
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

list(workflow.get_state_history(config1))

workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc72-ca16-6359-8001-7eea05e07dd2"}})

# above will start from previous step but this time the state will be different